# Chunk Summarization — Book-to-Moral pipeline (summarization phase)

For every `.txt` book in an input directory, this notebook produces two summaries
with a single GPT model:

1. **chunk_summary** — the full text is split into fixed-size pieces ("chunks")
   using **LLM tokenization** (`tiktoken`); each chunk is summarized as its ten
   most significant plot events, and the chunk summaries are concatenated.
2. **short_summary** — the chunk summary is condensed into one paragraph.

**Input:** a folder of `.txt` files (one book per file).

**Output:** a CSV summary table with one row per book and three columns:
`filename | chunk_summary | short_summary`. Only the filename is recorded —
join any metadata (author, year, language, …) to it later.

### How to use
Edit the **Parameters** cell, then run the notebook top to bottom
(`Kernel → Restart & Run All`). Every setting lives in the notebook — there are
no command-line arguments. The notebook checkpoints to the CSV after each book
and skips books already present on a rerun.

## 1. Imports

In [1]:
import time
from getpass import getpass
from pathlib import Path

import pandas as pd
import tiktoken
from openai import OpenAI

## 2. Parameters — edit these

Everything you might want to change lives here.

In [8]:
# --- Paths -------------------------------------------------------------------
INPUT_DIR  = Path("InputTexts")        # folder of .txt books
OUTPUT_CSV = Path("summaries.csv")     # where the summary table is written

# --- Model -------------------------------------------------------------------
# The single GPT model used for BOTH the chunk and the short summary.
MODEL = "gpt-5.4-mini"

# --- Output language ---------------------------------------------------------
# Injected into BOTH prompts below, so changing it changes the language the
# model writes the summaries in. Default English.
LANGUAGE = "English"

# --- Generation settings -----------------------------------------------------
TEMPERATURE      = 0       # 0 = deterministic
USE_TEMPERATURE  = True    # set False for reasoning models that reject `temperature`
CHUNK_MAX_TOKENS = 2000    # max output tokens per per-chunk summary
SHORT_MAX_TOKENS = 2000    # max output tokens for the one-paragraph condensation

# --- Chunking (LLM tokenization) ---------------------------------------------
TOKEN_ENCODING    = "o200k_base"   # tiktoken encoding (GPT-4o / GPT-5 family)
CHUNK_SIZE_TOKENS = 14000          # tokens per chunk

# --- Robustness / rate limiting ----------------------------------------------
DELAY_SECONDS   = 0.1      # pause between API calls
MAX_RETRIES     = 5        # retries on transient (429 / 5xx) errors
REQUEST_TIMEOUT = 120      # seconds per request
SAVE_EVERY      = 1        # write the CSV after every N books (checkpoint)

## 3. Prompts

The output `LANGUAGE` is injected into both prompts (upper-cased to match the
emphatic "IN ENGLISH" style). Edit the wording here if you want different
summary behavior.

In [4]:
# System role for the per-chunk summarization step.
CHUNK_SYSTEM_PROMPT = "You are a helpful assistant that summarizes literature."

def build_chunk_prompt(chunk_text: str, language: str) -> str:
    """User prompt for summarizing one chunk."""
    return (
        f"Here is a portion of a novel. Please summarize it IN {language.upper()} "
        "by listing the TEN most significant events and plot developments:\n\n"
        f"{chunk_text}"
    )

def build_short_prompt(language: str) -> str:
    """System prompt for the short-summary (condenser) step.
    The chunk summary is passed as the user message."""
    return (
        f"Please provide a 1 paragraph summary IN {language.upper()} "
        "of the following plot events of a novel:"
    )

## 4. API key

Requested live at runtime via a hidden prompt — the key is never hard-coded or
saved to disk. (Run this cell and paste your OpenAI key.)

In [5]:
api_key = getpass("Enter your OpenAI API key: ")
client = OpenAI(api_key=api_key)
encoding = tiktoken.get_encoding(TOKEN_ENCODING)
print("Client and tokenizer ready.")

Enter your OpenAI API key:  ········


Client and tokenizer ready.


## 5. Helper functions

In [6]:
def read_text_file(path: Path) -> str:
    """Read a .txt file as UTF-8, tolerant of bad bytes."""
    try:
        return path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        return path.read_text(encoding="utf-8", errors="replace")


def chunk_text_by_tokens(text: str, chunk_size_tokens: int = CHUNK_SIZE_TOKENS) -> list[str]:
    """Split text into fixed-size chunks using real LLM tokenization.

    The full text is encoded to token ids, sliced into windows of
    `chunk_size_tokens`, and each window is decoded back to text.
    """
    token_ids = encoding.encode(text)
    chunks = []
    for start in range(0, len(token_ids), chunk_size_tokens):
        chunk_ids = token_ids[start:start + chunk_size_tokens]
        chunks.append(encoding.decode(chunk_ids))
    return chunks


def call_gpt(system_prompt: str, user_prompt: str, max_tokens: int) -> str:
    """One chat-completion call, with retry/back-off on transient errors."""
    kwargs = dict(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_completion_tokens=max_tokens,
        timeout=REQUEST_TIMEOUT,
    )
    if USE_TEMPERATURE:
        kwargs["temperature"] = TEMPERATURE

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.chat.completions.create(**kwargs)
            return resp.choices[0].message.content
        except Exception as e:
            # Retry transient errors (rate limits, timeouts, 5xx) with back-off.
            transient = any(s in str(e).lower() for s in
                            ("429", "500", "502", "503", "529", "timeout", "overloaded"))
            if attempt == MAX_RETRIES or not transient:
                raise
            wait = 2 ** attempt
            print(f"    API error ({e}); retrying in {wait}s ({attempt}/{MAX_RETRIES})")
            time.sleep(wait)


def make_chunk_summary(full_text: str, language: str = LANGUAGE) -> str:
    """Stage A — summarize the full text chunk by chunk and concatenate."""
    chunks = chunk_text_by_tokens(full_text)
    print(f"  {len(chunks)} chunk(s)")
    summaries = []
    for i, chunk in enumerate(chunks, 1):
        print(f"    summarizing chunk {i}/{len(chunks)}")
        summaries.append(call_gpt(CHUNK_SYSTEM_PROMPT,
                                  build_chunk_prompt(chunk, language),
                                  CHUNK_MAX_TOKENS))
        time.sleep(DELAY_SECONDS)
    return "\n\n".join(summaries)


def make_short_summary(chunk_summary: str, language: str = LANGUAGE) -> str:
    """Stage B — condense the chunk summary into a single paragraph."""
    out = call_gpt(build_short_prompt(language), chunk_summary, SHORT_MAX_TOKENS)
    time.sleep(DELAY_SECONDS)
    return out

## 6. Main loop — process every book (with checkpoint / resume)

Processes each `.txt` file, writes `OUTPUT_CSV` after every `SAVE_EVERY` books,
and on a rerun skips any book already present in the CSV.

In [10]:
# Discover the books.
files = sorted(INPUT_DIR.glob("*.txt"))
if not files:
    raise FileNotFoundError(f"No .txt files found in: {INPUT_DIR}")
print(f"Found {len(files)} book(s) in {INPUT_DIR}")

# Resume: load any existing output and skip books already summarized.
if OUTPUT_CSV.exists():
    results = pd.read_csv(OUTPUT_CSV)
    print(f"Resuming — {len(results)} book(s) already in {OUTPUT_CSV}")
else:
    results = pd.DataFrame(columns=["filename", "chunk_summary", "short_summary"])

done = set(results["filename"].astype(str))
n_ok = 0
for path in files:
    book_id = path.stem                      # identifier = filename without .txt
    if book_id in done:
        print(f"SKIP  {book_id} (already summarized)")
        continue

    print(f"BOOK  {book_id}")
    try:
        full_text = read_text_file(path)
        chunk_summary = make_chunk_summary(full_text, LANGUAGE)
        short_summary = make_short_summary(chunk_summary, LANGUAGE)
        results.loc[len(results)] = [book_id, chunk_summary, short_summary]
        done.add(book_id)
        n_ok += 1
        if n_ok % SAVE_EVERY == 0:
            results.to_csv(OUTPUT_CSV, index=False)
            print("  checkpoint saved")
    except Exception as e:
        print(f"  ERROR on {book_id}: {e}")

# Final write.
results.to_csv(OUTPUT_CSV, index=False)
print(f"\nDone. {len(results)} book(s) in {OUTPUT_CSV}")

Found 1 book(s) in InputTexts
BOOK  Heart of Darkness
  4 chunk(s)
    summarizing chunk 1/4
    summarizing chunk 2/4
    summarizing chunk 3/4
    summarizing chunk 4/4
  checkpoint saved

Done. 1 book(s) in summaries.csv


## 7. Inspect the results

In [ ]:
results